# Advance the ingestion watermark

Attach `lh_meridian_hr` as the default lakehouse. In Fabric, mark Cell 2 as the parameter cell so the pipeline can override the values when advancing the watermark.

This notebook runs as the **last** activity of the pipeline and only advances the watermark. Creating and seeding `bronze.ingestion_watermark` is handled once, up front, by `nb_setup_lakehouse`.

## Parameters (mark this as the parameter cell)

**Summary.** Declares the two values the pipeline overrides at run time: which pipeline the watermark belongs to, and the timestamp to store.

<details>
<summary>Line-by-line details</summary>

- `pipeline_name = "workforce_events"` — the key that identifies this pipeline's row in the watermark table.
- `watermark_timestamp = "2020-12-01 00:00:00"` — the value to store; the pipeline passes the `watermark` returned by the discovery notebook.

</details>

In [ ]:
pipeline_name = "workforce_events"
watermark_timestamp = "2020-12-01 00:00:00"

## Advance the watermark

**Summary.** Validates the parameters, then MERGEs the new timestamp into the single row for this pipeline in `bronze.ingestion_watermark`.

<details>
<summary>Line-by-line details</summary>

- `from datetime import datetime` — used to parse and validate the supplied timestamp.
- The `if ... raise ValueError` guards reject an empty `pipeline_name` and a timestamp that is not the first day of a month.
- `spark.createDataFrame([...])` + `createOrReplaceTempView("watermark_input")` — build a one-row source for the MERGE.
- The MERGE updates `watermark_timestamp`/`updated_at` when the row exists; the `WHEN NOT MATCHED` insert is a safety net in case this notebook is run standalone before the setup notebook.
- The final `SELECT ... show(...)` prints the current watermark rows for confirmation.

</details>

In [ ]:
from datetime import datetime

if not pipeline_name.strip():
    raise ValueError("pipeline_name must not be empty")

parsed_watermark = datetime.strptime(watermark_timestamp, "%Y-%m-%d %H:%M:%S")
if parsed_watermark.day != 1:
    raise ValueError("watermark_timestamp must be the first day of a month")

watermark_input = spark.createDataFrame(
    [(pipeline_name, parsed_watermark)],
    "pipeline_name STRING, watermark_timestamp TIMESTAMP",
)
watermark_input.createOrReplaceTempView("watermark_input")

spark.sql("""
MERGE INTO bronze.ingestion_watermark AS target
USING watermark_input AS source
ON target.pipeline_name = source.pipeline_name
WHEN MATCHED THEN UPDATE SET
    target.watermark_timestamp = source.watermark_timestamp,
    target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
    VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
""")

spark.sql("SELECT * FROM bronze.ingestion_watermark ORDER BY pipeline_name").show(truncate=False)